# ML-02 — Research Question and Provisional Lane (Freestyle Direction)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dawngend/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook frames my **Freestyle Research Project** for the FlyRank Machine Learning Internship. As an aspiring Data Science / ML Specialist entering the industry job market, this project is designed to tackle a cutting-edge, real-world machine learning challenge: **Generative Engine Optimization (GEO) & AI Referral Traffic Intelligence** combined with multi-signal content performance modeling.

## 1. My lane (or freestyle) and why

**Lane Declaration:** **Freestyle Direction** — *AI Referral & Generative Engine Optimization (GEO) Opportunity Scoring: Multi-Signal Machine Learning for AI Search Traffic & Content Trajectories.*

**One-Paragraph Plan & Motivation:**
As an aspiring Data Science and Machine Learning professional, my goal during this 8-week internship is to build an industry-grade, resume-defining portfolio piece. Rather than restricting my work to standard SEO rules, I am pursuing a freestyle direction at the intersection of **AI Search Assistants** (ChatGPT, Perplexity, Claude, Gemini) and traditional organic performance. As user search behavior shifts toward conversational AI, web publishers face a dual challenge: protecting decaying organic search traffic while capturing sparse but high-intent AI referral traffic. By leveraging both the 30k starter dataset and the 78M+ warehouse dataset, I will build an end-to-end ML pipeline that scores content for **AI Visibility Gaps** (high search demand but under-indexed AI traffic) and predicts content performance trajectories using client-holdout cross-validation.

In [1]:
# Verification of Freestyle project parameters and dataset access
from pathlib import Path
import os

raw_path = Path('../../data/raw/content_refresh_anonymized.csv')
if not raw_path.exists():
    raw_path = Path('data/raw/content_refresh_anonymized.csv')

print('[✓] Project Mode: FREESTYLE (AI Referral & GEO Opportunity Scoring)')
print(f'[✓] Primary Data Source: {raw_path.resolve()}')
print(f'[✓] Dataset Exists & Ready: {raw_path.exists()}')


[✓] Project Mode: FREESTYLE (AI Referral & GEO Opportunity Scoring)
[✓] Primary Data Source: D:\Flyrank\FlyRank-Machine-Learning-Internship\data\raw\content_refresh_anonymized.csv
[✓] Dataset Exists & Ready: True


## 2. The question: decision, action, cost of a wrong call

### Problem Framing & Core Questions
- **Research Question:** *"Which high-demand content items exhibit an 'AI Visibility Gap' (strong search impressions but disproportionately low AI referral traffic), and how can machine learning models prioritize these pages for Generative Engine Optimization (GEO) and content decay protection?"*
- **Unit of Analysis:** A single pseudonymized content item (`content_id`) evaluated over a trailing 90-day window.
- **Decision Improved:** Deciding which specific articles an editorial and growth team should optimize for AI conversational engine discovery vs. standard organic search refresh.
- **Who Acts & Action Taken:** Content Editors, SEO Strategists, and AI Growth Engineers. Guided by ML scores and transparent reason codes (`ai_visibility_gap`, `high_demand_low_ai_referral`, `stale_visible_page`), they execute targeted interventions:
  - Structuring content with clear entity definitions, bulleted summaries, and FAQ schema for LLM retrieval (`ai_visibility_gap`)
  - Rewriting meta tags and updating outdated figures (`stale_visible_page` / `low_ctr_visible_page`)
  - Expanding thin articles with high organic/AI upside (`thin_visible_page`)
- **Cost of a Wrong Call:**
  - **False Positive (flagging a page with zero AI potential):** Wastes 3–5 hours of editorial/engineering time restructuring content that AI engines will not cite.
  - **False Negative (missing a high-potential AI opportunity):** Loss of market share and brand referral traffic as users transition from traditional Google searches to AI engine queries.
- **Why Data & ML Help:** AI-referred traffic is sparse (~6.4% of total pages) and exhibits complex non-linear relationships with search volume, position, word count, and engagement rate. Traditional heuristic rules fail on sparse signals, whereas machine learning models (e.g., Random Forest, Gradient Boosting with class weighting) effectively isolate high-probability candidates from background noise.

In [2]:
# Formal definition of Freestyle Decision Matrix for ML Resume Portfolio
freestyle_matrix = {
    'Target Persona': 'Growth Lead / AI SEO Strategist',
    'Unit of Analysis': 'content_id (pseudonymized content item)',
    'Primary ML Task': 'Imbalanced Classification & Priority Ranking (Precision@K)',
    'Key Target Signals': 'ai_sessions_90d, ai_traffic_pct, trend_direction',
    'Primary Action': 'Generative Engine Optimization (GEO) & Content Refresh'
}

for k, v in freestyle_matrix.items():
    print(f'{k:24s}: {v}')


Target Persona          : Growth Lead / AI SEO Strategist
Unit of Analysis        : content_id (pseudonymized content item)
Primary ML Task         : Imbalanced Classification & Priority Ranking (Precision@K)
Key Target Signals      : ai_sessions_90d, ai_traffic_pct, trend_direction
Primary Action          : Generative Engine Optimization (GEO) & Content Refresh


## 3. Quick look at the data (2-3 real numbers)

Evidence comes from two places, and I keep them clearly separated because they disagree in a way
that matters: the 30,000-row starter slice (`data/raw/content_refresh_anonymized.csv`) and the gated
warehouse release (`FlyRank/internship-warehouse`, 81.7M rows across four tables).

### 3.1 Starter slice, 30,000 items across 32 clients

1. **AI traffic is present but thin.** 1,930 items (6.43%) receive any AI referral traffic
   (`ai_sessions_90d` above zero), totalling 6,135 AI sessions. Where AI traffic exists at all it is
   material, averaging 11.94% of that page's sessions.
2. **The AI visibility gap.** 16,726 items (55.75%) have real organic demand
   (`impressions_90d` of at least 500), and 15,033 of those (89.88%) receive zero AI referral
   traffic. That population is the thing my lane proposes to rank.
3. **Decay runs alongside it.** 16,262 items (54.21%) carry a downward organic trend, so AI
   discovery and content decay are two pressures on the same inventory rather than separate problems.

### 3.2 The warehouse corrects the headline rate, and I would rather state that than hide it

Measured on the `month=2026-03` partition (9,841,378 daily rows), not quoted from documentation:

| Measure | Value |
|---|---:|
| Rows where GA4 is available at all | 413,966 (4.21%) |
| Rows with `sessions_ai` of at least 1 | 5,534 |
| ... as a share of GA4-available rows | **1.34%** |
| Content items with any AI traffic | 3,795 of 331,437 (1.15%) |
| Clients with any AI traffic | 35 |

**Reconciling 6.43% with 1.15%.** The starter figure counts items over a 90-day window in a curated
32-client slice. The warehouse figure counts items over a single 31-day month across 55 clients.
Different windows, different populations, so the honest reading is that the starter slice
overstates AI density by roughly 5x rather than that either number is wrong. The denominator matters
too: quoting AI presence against *all* daily rows gives 0.056%, but 95.8% of those rows have no GA4
tracking at all, so AI traffic could never have been observed there. Against rows where it could be
observed, the rate is 1.34%.

**Why the lane survives this.** 3,795 positive items in one month is a workable ranking target. It
is imbalanced enough that Precision@K is the right metric and accuracy would be meaningless, which
is exactly the framing carried into ML-03 and ML-04.

### 3.3 The engine-level split, which the starter dataset cannot show

`fact_content_daily_performance` carries per-engine referral columns. March 2026:

| Engine | Sessions | Share of AI referrals |
|---|---:|---:|
| ChatGPT | 5,155 | 57.85% |
| Gemini | 2,527 | 28.36% |
| Perplexity | 970 | 10.89% |
| Copilot | 144 | 1.62% |
| Claude | 118 | 1.32% |

The starter CSV holds only a single `ai_sessions_90d` total, so this is invisible there. Two engines
account for 86% of AI referrals, which turns "optimize for AI search" from a slogan into a
measurable, targetable question. `ai_meta` and `ai_other` are zero throughout this partition, so
they are constant columns for this month and must not be fed to a model as if they carried signal.

### 3.4 The bar my lane has to clear

The repository ships a runnable starter pipeline (`scripts/run_all.py`) whose target is the proxy
label `is_declining_label = trend_direction == "down"`. Run end to end on a client-holdout split it
produces a rule baseline at **Precision@50 = 0.24** and a random forest at **Precision@50 = 0.68**,
a 2.83x lift, with ROC-AUC 0.747.

**These are the shipped example's numbers on the shipped example's target, not my results and not my
lane.** I record them here because they are the reference point my freestyle work has to be measured
against, and because a 0.68 Precision@50 on a 54% base rate is a much easier problem than ranking a
1.15% positive class. Any number I report later will be stated against its own base rate so the two
are never confused.


In [3]:
# Section 3 - evidence. Starter slice measured locally, warehouse measured live when a token exists.
import json
import os
from pathlib import Path

import pandas as pd

def find_starter_csv() -> Path:
    for candidate in (
        Path("data/raw/content_refresh_anonymized.csv"),
        Path("../../data/raw/content_refresh_anonymized.csv"),
        Path("/content/FlyRank-Machine-Learning-Internship/data/raw/content_refresh_anonymized.csv"),
    ):
        if candidate.exists():
            return candidate
    raise FileNotFoundError("starter CSV not found; see SETUP.md")

df = pd.read_csv(find_starter_csv())
MIN_IMPRESSIONS = 500

total = len(df)
ai_active = int((df["ai_sessions_90d"] > 0).sum())
high_demand = df["impressions_90d"] >= MIN_IMPRESSIONS
gap = high_demand & (df["ai_sessions_90d"] == 0)
declining = int((df["trend_direction"] == "down").sum())

print("=" * 72)
print("3.1 STARTER SLICE")
print("=" * 72)
print(f"  inventory                 : {total:,} items across {df['client_id'].nunique()} clients")
print(f"  items with AI traffic     : {ai_active:,} ({ai_active / total * 100:.2f}%), "
      f"{int(df['ai_sessions_90d'].sum()):,} AI sessions total")
print(f"  mean AI share when present: {df.loc[df['ai_sessions_90d'] > 0, 'ai_traffic_pct'].mean():.2f}%")
print(f"  high-demand items         : {int(high_demand.sum()):,} ({high_demand.mean() * 100:.2f}%)")
print(f"  ... of those, zero AI     : {int(gap.sum()):,} ({gap.sum() / high_demand.sum() * 100:.2f}%)")
print(f"  declining organic trend   : {declining:,} ({declining / total * 100:.2f}%)")

# --- Warehouse -------------------------------------------------------------
BASE = "hf://datasets/FlyRank/internship-warehouse"
MAR = f"{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet"

RECORDED = {  # measured 2026-08-19; re-measured live whenever HF_TOKEN is present
    "daily_rows": 9_841_378, "ga4_rows": 413_966, "ai_rows": 5_534,
    "ai_items": 3_795, "items": 331_437, "ai_clients": 35,
    "engines": {"chatgpt": 5_155, "gemini": 2_527, "perplexity": 970,
                "copilot": 144, "claude": 118, "meta": 0, "other": 0},
}

live = False
try:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("HF_TOKEN not set")
    import duckdb
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute("CREATE SECRET (TYPE HUGGINGFACE, PROVIDER credential_chain);")
    try:
        con.execute("SET enable_progress_bar=false;")   # cosmetic; never fatal
    except Exception:
        pass
    row = con.execute(
        "SELECT COUNT(*), SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END), "
        "SUM(CASE WHEN sessions_ai>=1 THEN 1 ELSE 0 END), "
        "COUNT(DISTINCT CASE WHEN sessions_ai>=1 THEN content_hash_id END), "
        "COUNT(DISTINCT content_hash_id), "
        "COUNT(DISTINCT CASE WHEN sessions_ai>=1 THEN client_hash_id END) "
        f"FROM '{MAR}'").fetchone()
    eng = con.execute(
        "SELECT SUM(ai_chatgpt), SUM(ai_gemini), SUM(ai_perplexity), SUM(ai_copilot), "
        f"SUM(ai_claude), SUM(ai_meta), SUM(ai_other) FROM '{MAR}'").fetchone()
    RECORDED.update(dict(zip(["daily_rows", "ga4_rows", "ai_rows", "ai_items", "items", "ai_clients"], row)))
    RECORDED["engines"] = dict(zip(["chatgpt", "gemini", "perplexity", "copilot", "claude", "meta", "other"],
                                   [v or 0 for v in eng]))
    live = True
except Exception as exc:
    print(f"\n  [warehouse not queried] {type(exc).__name__}: {exc}")
    print("  Using the values recorded on 2026-08-19; set HF_TOKEN to re-measure.")

w = RECORDED
print("\n" + "=" * 72)
print(f"3.2 WAREHOUSE, month=2026-03  [{'live' if live else 'recorded'}]")
print("=" * 72)
print(f"  daily rows                : {w['daily_rows']:,}")
print(f"  GA4 available             : {w['ga4_rows']:,} ({w['ga4_rows'] / w['daily_rows'] * 100:.2f}%)")
print(f"  rows with sessions_ai     : {w['ai_rows']:,}")
print(f"    share of ALL rows       : {w['ai_rows'] / w['daily_rows'] * 100:.4f}%   <- wrong denominator")
print(f"    share of GA4-available  : {w['ai_rows'] / w['ga4_rows'] * 100:.2f}%   <- the honest one")
print(f"  items with any AI traffic : {w['ai_items']:,} of {w['items']:,} "
      f"({w['ai_items'] / w['items'] * 100:.2f}%)")
print(f"  clients with any AI       : {w['ai_clients']}")
print(f"\n  starter {ai_active / total * 100:.2f}% of items over 90 days vs warehouse "
      f"{w['ai_items'] / w['items'] * 100:.2f}% over 31 days: about 5x, different windows and populations.")

print("\n" + "=" * 72)
print("3.3 ENGINE SPLIT (invisible in the starter dataset)")
print("=" * 72)
eng_total = sum(w["engines"].values())
for name, val in sorted(w["engines"].items(), key=lambda kv: -kv[1]):
    note = "   <- constant zero this month, do not model" if val == 0 else ""
    print(f"  {name:12s} {val:>8,}  ({val / eng_total * 100 if eng_total else 0:5.2f}%){note}")
top2 = sum(sorted(w["engines"].values(), reverse=True)[:2]) / eng_total * 100 if eng_total else 0
print(f"  top two engines account for {top2:.1f}% of AI referrals")

print("\n" + "=" * 72)
print("3.4 THE BAR: the shipped starter pipeline, NOT my lane and NOT my result")
print("=" * 72)
results_path = next((p for p in [Path("outputs/model_results.json"),
                                 Path("../../outputs/model_results.json")] if p.exists()), None)
if results_path:
    res = json.loads(results_path.read_text(encoding="utf-8"))
    base = res["baseline"]["baseline_precision_at_50"]
    rf = res["models"]["random_forest"]["precision_at_50"]
    print(f"  source                : {results_path} (produced by scripts/run_all.py)")
    print(f"  target                : is_declining_label = trend_direction == 'down'")
    print(f"  positive base rate    : {declining / total * 100:.2f}%   <- an easy problem")
    print(f"  rule baseline P@50    : {base:.2f}")
    print(f"  random forest P@50    : {rf:.2f}   ({rf / base:.2f}x lift)")
    print(f"  random forest ROC-AUC : {res['models']['random_forest']['roc_auc']:.3f}")
else:
    print("  outputs/model_results.json not present; run `python scripts/run_all.py` to regenerate.")
    print("  Recorded on 2026-08-19: baseline P@50 0.24, random forest P@50 0.68, 2.83x lift.")
print(f"\n  My lane ranks a ~{w['ai_items'] / w['items'] * 100:.2f}% positive class, not a "
      f"{declining / total * 100:.0f}% one. Precision@K numbers are only comparable")
print("  against their own base rate, so every metric I report later will carry it.")


3.1 STARTER SLICE
  inventory                 : 30,000 items across 32 clients
  items with AI traffic     : 1,930 (6.43%), 6,135 AI sessions total
  mean AI share when present: 11.94%
  high-demand items         : 16,726 (55.75%)
  ... of those, zero AI     : 15,033 (89.88%)
  declining organic trend   : 16,262 (54.21%)



3.2 WAREHOUSE, month=2026-03  [live]
  daily rows                : 9,841,378
  GA4 available             : 413,966 (4.21%)
  rows with sessions_ai     : 5,534
    share of ALL rows       : 0.0562%   <- wrong denominator
    share of GA4-available  : 1.34%   <- the honest one
  items with any AI traffic : 3,795 of 331,437 (1.15%)
  clients with any AI       : 35

  starter 6.43% of items over 90 days vs warehouse 1.15% over 31 days: about 5x, different windows and populations.

3.3 ENGINE SPLIT (invisible in the starter dataset)
  chatgpt         5,155  (57.83%)
  gemini          2,527  (28.35%)
  perplexity        970  (10.88%)
  copilot           144  ( 1.62%)
  claude            118  ( 1.32%)
  meta                0  ( 0.00%)   <- constant zero this month, do not model
  other               0  ( 0.00%)   <- constant zero this month, do not model
  top two engines account for 86.2% of AI referrals

3.4 THE BAR: the shipped starter pipeline, NOT my lane and NOT my result
  source     

## 4. Careful words: what I can and can't claim

### What I can claim

- **Observed association.** Measured relationships between search and engagement signals
  (impressions, clicks, position, word count, engagement) and the presence of AI referral sessions,
  over stated windows on a pseudonymized panel.
- **Relative prioritization.** A ranked review queue that surfaces high-demand pages with
  disproportionately low AI referral traffic, judged by Precision@K reported beside its base rate.
- **Engine-level description.** That AI referral volume in this panel is concentrated in ChatGPT and
  Gemini, because `ai_chatgpt` and `ai_gemini` are measured columns. This describes the traffic,
  never the retrieval behaviour that produced it.
- **Decision support.** That editor time is better spent on a ranked queue than on an unordered
  inventory of 30,000 pages, judged against the shipped rule baseline.

### What I cannot claim

- **Causation.** No content edit is observed in this data. A rise after a change cannot be attributed
  to that change without a controlled experiment.
- **Guaranteed citation.** That structuring a page for generative engines makes any AI assistant cite
  or link it.
- **LLM internals.** That this work inspects weights, prompts, or retrieval mechanisms of ChatGPT,
  Gemini, Perplexity, Copilot or Claude. The per-engine columns show *that* referral volume differs
  by engine, never *why*.
- **Anything about real pages, clients or queries.** Every identifier is a pseudonym.
- **Other people's results as my own.** The 0.24 and 0.68 Precision@50 figures in section 3.4 belong
  to the repository's shipped starter pipeline on the starter proxy label. They are the bar, not my
  finding, and are labelled that way wherever they appear.
- **Population generality.** The panel is unbalanced and only 54 of 104 clients carry the GA4 access
  this lane needs, so a finding here describes this panel, not the open web.


In [4]:
# Section 4 - claim boundaries, kept as a checkable object rather than prose alone.
CLAIM_BOUNDARIES = {
    "supportable": [
        "observed association between search/engagement signals and AI referral presence",
        "relative prioritization via Precision@K reported beside its base rate",
        "engine-level description of AI referral volume (measured columns)",
        "decision support: a ranked queue beats an unordered 30k inventory",
    ],
    "not_supportable": [
        "causal attribution without a controlled experiment",
        "guaranteed AI citation from GEO work",
        "any statement about LLM internals, weights, prompts, or retrieval",
        "any claim about real pages, clients, or queries behind the pseudonyms",
        "presenting the shipped starter pipeline's metrics as my own results",
        "generalizing beyond this unbalanced 104-client panel",
    ],
}

ATTRIBUTION = {
    "baseline_p_at_50": {"value": 0.24, "owner": "repo starter pipeline", "target": "trend_direction == 'down'"},
    "random_forest_p_at_50": {"value": 0.68, "owner": "repo starter pipeline", "target": "trend_direction == 'down'"},
}

print("SUPPORTABLE")
for c in CLAIM_BOUNDARIES["supportable"]:
    print(f"  [ok] {c}")
print("\nNOT SUPPORTABLE")
for c in CLAIM_BOUNDARIES["not_supportable"]:
    print(f"  [no] {c}")

print("\nATTRIBUTION OF EVERY BORROWED NUMBER")
for key, meta in ATTRIBUTION.items():
    print(f"  {key:24s} = {meta['value']}  owner: {meta['owner']}  target: {meta['target']}")

assert all(m["owner"] != "me" for m in ATTRIBUTION.values()), \
    "a borrowed metric is being presented as an own result"
print("\n[OK] no borrowed metric is presented as my own result")


SUPPORTABLE
  [ok] observed association between search/engagement signals and AI referral presence
  [ok] relative prioritization via Precision@K reported beside its base rate
  [ok] engine-level description of AI referral volume (measured columns)
  [ok] decision support: a ranked queue beats an unordered 30k inventory

NOT SUPPORTABLE
  [no] causal attribution without a controlled experiment
  [no] guaranteed AI citation from GEO work
  [no] any statement about LLM internals, weights, prompts, or retrieval
  [no] any claim about real pages, clients, or queries behind the pseudonyms
  [no] presenting the shipped starter pipeline's metrics as my own results
  [no] generalizing beyond this unbalanced 104-client panel

ATTRIBUTION OF EVERY BORROWED NUMBER
  baseline_p_at_50         = 0.24  owner: repo starter pipeline  target: trend_direction == 'down'
  random_forest_p_at_50    = 0.68  owner: repo starter pipeline  target: trend_direction == 'down'

[OK] no borrowed metric is presented 

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.